In [1]:
# ============================================================
# Feature Engineering on Titanic Dataset
# ============================================================
# Author: Proshanta Pal
# Dataset: Titanic Dataset
# Date: 2026-05-21
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import joblib
import os

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print("All libraries loaded successfully!")

All libraries loaded successfully!


In [2]:
# Load the raw Data
df = pd.read_csv('../data/raw/titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
# First look at the data
print(f"Shape of Dataset: {df.shape}")
rows, cols = df.shape
print(f"Titanic Dataset has {rows} rows and {cols} columns.")

Shape of Dataset: (891, 12)
Titanic Dataset has 891 rows and 12 columns.


## Train-Test Split 
A train-test split is a machine learning technique where you divide your dataset into two subsets:
- a `training set` used to teach the model, and 
- a `testing set` used to evaluate its performance on unseen data

> `train_test_split` randomly shuffles and splits the data.

In [4]:
# Train-Test Split

# Separate our target column from features
# X = everything the model learns from (features)
# y = what model predicts (target)
X = df.drop(columns=['Survived']) 
y = df['Survived']

# test_size = 0.2 (20% goes to test), and 80% goes to training
# random_state = fixes the randomness so results are reproducible 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")

Training set: 712 rows
Test set: 179 rows


In [5]:
# Fill Structural Missing Values
# These are the columns where data 'does not exist'

X_train = X_train.copy()
X_test = X_test.copy()

# Cabin column has nearly 77% missing data, so fill it with 'No_Cabin'
X_train['Cabin'] = X_train['Cabin'].fillna('No_Cabin')
X_test['Cabin'] = X_test['Cabin'].fillna('No_Cabin')

print("Cabin - missing value check after fill:")
print(f"Training set: {X_train['Cabin'].isnull().sum()} missing out of {len(X_train)} rows")
print(f"Testing set : {X_test['Cabin'].isnull().sum()} missing out of {len(X_test)} rows")

Cabin - missing value check after fill:
Training set: 0 missing out of 712 rows
Testing set : 0 missing out of 179 rows


In [6]:
# Fill Statistical Missing Values
# These are the columns where the data is truly absent and must be estimated

# Embarked column has only 2 missing values, so fill it with Mode 
embarked_mode = X_train['Embarked'].mode()[0]
print(f"Embarked mode (only from training set): {embarked_mode}")

X_train['Embarked'] = X_train['Embarked'].fillna(embarked_mode)
X_test['Embarked'] = X_test['Embarked'].fillna(embarked_mode)

# Age column has nearly 19.87% missing data, so fill it with Median
# Use median because Age is slightly right-skewed
age_median = X_train['Age'].median()
print(f"Age median (only from training set): {age_median}", end="\n\n")

X_train['Age'] = X_train['Age'].fillna(age_median)
X_test['Age'] = X_test['Age'].fillna(age_median)

print("Missing values remaining in X_train: ", end="")
remaining = X_train.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0  else "None - Clean!")

Embarked mode (only from training set): S
Age median (only from training set): 28.0

Missing values remaining in X_train: None - Clean!


In [7]:
# Final Missing Value Verification (if fillna don't work)
print("X_train missing values after all fills:")
train_missing = X_train.isnull().sum()
print(train_missing[train_missing > 0] if train_missing.sum() > 0 else "No missing values in training set")

print()
print("X_test missing values after all fills:")
test_missing = X_test.isnull().sum()
print(test_missing[test_missing > 0] if test_missing.sum() > 0 else "No missing values in testingg set")

X_train missing values after all fills:
No missing values in training set

X_test missing values after all fills:
No missing values in testingg set


## Encoding Categorical Variables

In [8]:
# Binary Encoding (Sex Column)

# We map female = 1, male = 0
# map() replaces each value using the dictionary
sex_map = {'female': 1, 'male': 0}

X_train['Sex'] = X_train['Sex'].map(sex_map)
X_test['Sex'] = X_test['Sex'].map(sex_map)

print("Sex column - unique values after encoding:")
print(f"Training set: {X_train['Sex'].unique()}")
print(f"Testing set: {X_test['Sex'].unique()}")

print("Value counts in training set:")
print(X_train['Sex'].value_counts())

Sex column - unique values after encoding:
Training set: [0 1]
Testing set: [0 1]
Value counts in training set:
Sex
0    467
1    245
Name: count, dtype: int64


In [9]:
# Ordinal Encoding (Pclass Column)

# 1st Class = 3 (highest class)
# 2nd Class = 2 
# 3rd Class = 1 (Lowest class)
pclass_map = {1: 3, 2: 2, 3: 1}

X_train['Pclass'] = X_train['Pclass'].map(pclass_map)
X_test['Pclass'] = X_test['Pclass'].map(pclass_map)

print("Pclass after Ordinal Encoding (inverted):")
print(X_train['Pclass'].value_counts().sort_index())

Pclass after Ordinal Encoding (inverted):
Pclass
1    398
2    151
3    163
Name: count, dtype: int64


In [10]:
# One-Hot Encoding (Embarked Column)

# Encode the train set
train_embarked = pd.get_dummies(X_train['Embarked'], prefix='Embarked', drop_first=True)

# Encode the test set
test_embarked = pd.get_dummies(X_test['Embarked'], prefix='Embarked', drop_first=True)

# Join the new columns to our dataset
X_train = pd.concat([X_train, train_embarked], axis=1)
X_test = pd.concat([X_test, test_embarked], axis=1)

# Drop the original Embarked column 
X_train = X_train.drop(columns=['Embarked'])
X_test = X_test.drop(columns=['Embarked'])

print("New Embarked columns added:")
embarked_cols = [c for c in X_train.columns if 'Embarked' in c]
print(f"{embarked_cols}", end='\n\n')

New Embarked columns added:
['Embarked_Q', 'Embarked_S']



In [11]:
X_train.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked_Q,Embarked_S
331,332,3,"Partner, Mr. Austen",0,45.5,0,0,113043,28.5000,C124,False,True
733,734,2,"Berriman, Mr. William John",0,23.0,0,0,28425,13.0000,No_Cabin,False,True
382,383,1,"Tikkanen, Mr. Juho",0,32.0,0,0,STON/O 2. 3101293,7.9250,No_Cabin,False,True
704,705,1,"Hansen, Mr. Henrik Juul",0,26.0,1,0,350025,7.8542,No_Cabin,False,True
813,814,1,"Andersson, Miss. Ebba Iris Alfrida",1,6.0,4,2,347082,31.2750,No_Cabin,False,True


In [12]:
X_test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked_Q,Embarked_S
709,710,1,"Moubarek, Master. Halim Gonios (""William George"")",0,28.0,1,1,2661,15.2458,No_Cabin,False,False
439,440,2,"Kvillner, Mr. Johan Henrik Johannesson",0,31.0,0,0,C.A. 18723,10.5000,No_Cabin,False,True
840,841,1,"Alhomaki, Mr. Ilmari Rudolf",0,20.0,0,0,SOTON/O2 3101287,7.9250,No_Cabin,False,True
720,721,2,"Harper, Miss. Annie Jessie ""Nina""",1,6.0,0,1,248727,33.0000,No_Cabin,False,True
39,40,1,"Nicola-Yarred, Miss. Jamila",1,14.0,1,0,2651,11.2417,No_Cabin,False,False


In [13]:
# Drop Identifiers and Unusual Columns
cols_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin']

X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

print("Remaining columns after dropping identifiers:")
print(X_train.columns.tolist(), end='\n\n')
print(f"Shape - Training set: {X_train.shape} and Testoing set: {X_test.shape}")

Remaining columns after dropping identifiers:
['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_Q', 'Embarked_S']

Shape - Training set: (712, 8) and Testoing set: (179, 8)


## Feature Scaling
**Never Scale:**
- Binary columns - (already 0 and 1: scaling breaks their meaning)
- One-hot encoded columns
- The target column y

In [14]:
# Step 1: Identify which columns need scaling
# Only scale numerical continuous/discrete columns
cols_to_scale = ['Age', 'Fare']

# Step 2: Create a Scaler object
scaler = StandardScaler()

# Step 3: Fit the training set only (learns mean and std from X_train)
scaler.fit(X_train[cols_to_scale])

# Step 4: Transform both train and test using the fitted scaler
# .transform() applies the formula: z = (x - mean) / std
X_train[cols_to_scale] = scaler.transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print("After StandardScaler:", end='\n\n')
print(f"Age (mean): {X_train['Age'].mean():.4f}, (std): {X_train['Age'].std():.4f}")
print(f"Fare (mean): {X_train['Fare'].mean():.4f}, (std): {X_train['Fare'].std():.4f}")

After StandardScaler:

Age (mean): 0.0000, (std): 1.0007
Fare (mean): 0.0000, (std): 1.0007


In [18]:
# ddof=0 means divide by N - matches what StandardScaler used
print(f"Age (mean): {X_train['Age'].mean():.6f}, (std): {X_train['Age'].std(ddof=0):.6f}")
print(f"Fare (mean): {X_train['Fare'].mean():.6f}, (std): {X_train['Fare'].std(ddof=0):.6f}")

Age (mean): 0.000000, (std): 1.000000
Fare (mean): 0.000000, (std): 1.000000


In [15]:
# Save the Scaler for Later Use

# Create a models folder to store trained objects
os.makedirs('../models', exist_ok=True)

# joblib.dump() serializes the Python object to disk
# A serialized scaler remembers the exact mean and std
joblib.dump(scaler, '../models/standard_scaler.pkl')

print("Saved Successfully")
print(f"Scaler learned mean: {scaler.mean_}")
print(f"Scaler learned std: {scaler.scale_}")

Saved Successfully
Scaler learned mean: [29.20412921 32.58627612]
Scaler learned std: [12.998833   51.93302107]


In [16]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
331,3,0,1.253641,0,0,-0.078684,False,True
733,2,0,-0.477284,0,0,-0.377145,False,True
382,1,0,0.215086,0,0,-0.474867,False,True
704,1,0,-0.246494,1,0,-0.476230,False,True
813,1,1,-1.785093,4,2,-0.025249,False,True


In [17]:
X_test.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
709,1,0,-0.092634,1,1,-0.333901,False,False
439,2,0,0.138156,0,0,-0.425284,False,True
840,1,0,-0.708074,0,0,-0.474867,False,True
720,2,1,-1.785093,0,1,0.007966,False,True
39,1,1,-1.169653,1,0,-0.411002,False,False
